# LeetCode 1935 — Maximum Number of Words You Can Type bằng Python

**Link đề:** https://leetcode.com/problems/maximum-number-of-words-you-can-type/
**Độ khó:** Easy · **Chủ đề:** Hash Table, String

```python
class Solution:
    def canBeTypedWords(self, text: str, brokenLetters: str) -> int:
        ...
```

## Mục tiêu học
1. Biết tách câu thành từ bằng `str.split()` và duyệt từng ký tự.
2. Hiểu toán tử `in` trên `str` (chậm, duyệt tuần tự) và trên `set` (nhanh, tra băm) khác nhau thế nào.
3. Dùng `any` / `all`, phép giao tập `&`, `set.isdisjoint` để viết lời giải gọn mà vẫn hiểu bản chất.
4. Viết thêm lời giải **một lượt không cần `split`** với cờ (flag) trạng thái.
5. Trace, chứng minh đúng, phân tích Time/Space và luyện biến thể.

**Cách học:** đọc markdown → đoán output của cell code kế tiếp → chạy → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài — bàn phím hỏng vài phím chữ

Bàn phím bị hỏng một số phím chữ; các phím khác vẫn tốt. Cho:
- `text`: các từ ngăn cách bởi **đúng một dấu cách**, không có dấu cách ở đầu/cuối;
- `brokenLetters`: chuỗi các phím hỏng (các chữ cái **khác nhau**).

Trả về số từ trong `text` mà ta gõ **trọn vẹn** được (từ không chứa phím hỏng nào).

| `text` | `brokenLetters` | Output | Giải thích |
|---|---|:---:|---|
| `"hello world"` | `"ad"` | `1` | `"world"` chứa `d` |
| `"leet code"` | `"lt"` | `1` | `"leet"` chứa `l`, `t` |
| `"leet code"` | `"e"` | `0` | cả hai từ đều chứa `e` |

**Constraints:**
- `1 <= text.length <= 10^4`
- `0 <= brokenLetters.length <= 26` (có thể **rỗng** — khi đó gõ được mọi từ)
- mỗi từ chỉ gồm chữ thường; `brokenLetters` gồm chữ thường khác nhau.

Câu hỏi trước khi code: một từ "gõ được" nghĩa là gì, viết bằng logic? → *mọi* ký tự của từ đều **không** nằm trong `brokenLetters`.

In [1]:
# Kiểm tra bằng tay ví dụ 1
text, broken = "hello world", "ad"
for word in text.split():
    bad = [ch for ch in word if ch in broken]
    print(word, "-> chữ hỏng:", bad, "-> gõ được" if not bad else "-> KHÔNG gõ được")

hello -> chữ hỏng: [] -> gõ được
world -> chữ hỏng: ['d'] -> KHÔNG gõ được


## 2. Nền tảng Python cần dùng

### 2.1. `str.split()` — tách câu thành list các từ

`text.split()` (không đối số) tách theo khoảng trắng và **bỏ** chuỗi rỗng. `text.split(" ")` tách theo đúng một dấu cách — với đề này (đúng một dấu cách, không thừa) hai cách cho cùng kết quả, nhưng khác nhau khi dữ liệu "bẩn".

In [2]:
print("hello world".split())        # ['hello', 'world']
print("leet code".split(" "))       # ['leet', 'code']
print("a  b".split(), "a  b".split(" "))   # ['a', 'b'] ['a', '', 'b'] — khác nhau khi có 2 dấu cách
print("single".split())             # ['single'] — một từ vẫn là list 1 phần tử

['hello', 'world']
['leet', 'code']
['a', 'b'] ['a', '', 'b']
['single']


### 2.2. Toán tử `in` trên chuỗi và trên `set`

- `ch in "ad"`: Python duyệt chuỗi `"ad"` từ trái sang phải để tìm `ch` → tốn tối đa `len(brokenLetters)` bước (≤ 26).
- `ch in {'a', 'd'}`: `set` dùng **bảng băm** (hash table), tra cứu trung bình `O(1)`.

Ở bài này `brokenLetters` dài tối đa 26 nên cả hai đều nhanh, nhưng thói quen "chuyển sang `set` khi cần tra nhiều lần" rất quan trọng cho các bài lớn hơn.

In [3]:
broken_str = "ad"
broken_set = set(broken_str)
print(broken_set)                       # {'a', 'd'} (thứ tự in có thể khác)
print("d" in broken_str, "d" in broken_set)   # True True
print("z" in broken_str, "z" in broken_set)   # False False
print(set(""))                           # set() — tập rỗng khi không có phím hỏng

{'d', 'a'}
True True
False False
set()


### 2.3. `any` và `all`

- `any(điều_kiện for x in dãy)` → `True` nếu **có ít nhất một** phần tử thoả; dừng ngay khi gặp phần tử đầu tiên thoả (short-circuit).
- `all(...)` → `True` nếu **mọi** phần tử thoả; dừng ngay khi gặp phần tử sai.
- Với dãy rỗng: `any([]) == False`, `all([]) == True`.

Từ gõ được ⇔ `all(ch not in broken for ch in word)` ⇔ `not any(ch in broken for ch in word)`.

In [4]:
broken = {"a", "d"}
print(all(ch not in broken for ch in "hello"))   # True
print(any(ch in broken for ch in "world"))       # True (vì có 'd')
print(any([]), all([]))                          # False True

True
True
False True


### 2.4. Phép toán tập hợp: giao `&` và `isdisjoint`

- `A & B` là các phần tử **chung** của hai tập.
- `A.isdisjoint(B)` trả `True` khi hai tập **không có phần tử chung** (rời nhau) — không cần tạo tập giao.

Từ gõ được ⇔ `set(word)` và `broken` rời nhau.

In [5]:
broken = set("ad")
print(set("world") & broken)            # {'d'}
print(set("hello") & broken)            # set()
print(set("hello").isdisjoint(broken))  # True
print(set("world").isdisjoint(broken))  # False

{'d'}
set()
True
False


### 2.5. Biến đếm và `sum` trên giá trị bool

Trong Python, `True` hành xử như `1`, `False` như `0`. Vì vậy `sum(điều_kiện for ...)` đếm số phần tử thoả điều kiện.

In [6]:
print(True + True + False)                         # 2
words = ["hello", "world"]
print(sum(len(w) == 5 for w in words))             # 2
print(sum(w.startswith("w") for w in words))       # 1

2
2
1


## 3. Tính tay trước khi code

Ví dụ 2: `text = "leet code"`, `broken = "lt"`.

| Từ | Duyệt ký tự | Gặp ký tự hỏng? | Kết luận |
|---|---|---|---|
| `leet` | `l` | có (`l`) — dừng ngay | không gõ được |
| `code` | `c`,`o`,`d`,`e` | không | gõ được → đếm 1 |

Kết quả `1`. Để ý: với từ `leet` ta **không cần** xem hết các ký tự — gặp `l` là đủ kết luận. Đây là ý tưởng "dừng sớm" (`break`).

## 4. Cách 1 — Baseline: vòng lặp lồng, kiểm tra `in` trên chuỗi

Ý tưởng trực giác nhất: với mỗi từ, đặt cờ `ok = True`; duyệt từng ký tự, nếu ký tự nằm trong `brokenLetters` thì `ok = False` và `break`. Cuối mỗi từ, nếu `ok` vẫn `True` thì đếm.

In [7]:
def can_type_baseline(text, brokenLetters):
    count = 0
    for word in text.split():
        ok = True
        for ch in word:
            if ch in brokenLetters:     # 'in' trên chuỗi: duyệt tuần tự
                ok = False
                break                   # đã hỏng thì không cần xem tiếp
        if ok:
            count += 1
    return count

print(can_type_baseline("hello world", "ad"))  # 1
print(can_type_baseline("leet code", "lt"))    # 1
print(can_type_baseline("leet code", "e"))     # 0

1
1
0


### 4.1. `break` và vòng `for ... else`

`break` thoát khỏi **vòng lặp gần nhất** (vòng ký tự), không thoát vòng từ. Python còn có cú pháp `for ... else`: khối `else` chạy khi vòng lặp **không** bị `break`. Ta có thể bỏ biến cờ `ok`:

In [8]:
def can_type_for_else(text, brokenLetters):
    count = 0
    for word in text.split():
        for ch in word:
            if ch in brokenLetters:
                break
        else:                 # chỉ chạy khi vòng 'for ch' KHÔNG gặp break
            count += 1
    return count

print(can_type_for_else("hello world", "ad"), can_type_for_else("leet code", "e"))

1 0


### 4.2. Chi phí của baseline

Gọi `n = len(text)`, `m = len(brokenLetters)` (≤ 26). Mỗi ký tự của `text` có thể bị so với tới `m` phím hỏng → `O(n·m)`. Với `m ≤ 26` thực tế vẫn rất nhanh, nhưng về mặt kỹ thuật, phần `ch in brokenLetters` **lặp lại công việc tìm kiếm** cho mỗi ký tự.

## 5. Bước cải tiến — tra cứu bằng `set`

Nút thắt là phép tìm `ch` trong chuỗi phím hỏng. Chuyển `brokenLetters` thành `set` **một lần** (`O(m)`), sau đó mỗi phép tra cứu trung bình `O(1)` → tổng `O(n + m)`.

Ý tưởng không đổi: *một từ gõ được ⇔ không ký tự nào nằm trong tập phím hỏng*.

In [9]:
# Đếm số phép so sánh khi tra trong CHUỖI (duyệt tuần tự) — set thì chỉ 1 lần băm
def comparisons_in_string(ch, keys):
    steps = 0
    for k in keys:
        steps += 1
        if k == ch:
            break
    return steps

keys = "abcdefghijklmnopqrstuvwxyz"
print("tìm 'a':", comparisons_in_string("a", keys), "phép so sánh")
print("tìm 'z':", comparisons_in_string("z", keys), "phép so sánh")
print("tìm '#':", comparisons_in_string("#", keys), "phép so sánh (không có, phải duyệt hết)")
text = "zzz " * 1000
print("tổng so sánh cho cả text:", sum(comparisons_in_string(ch, keys) for ch in text if ch != " "))

tìm 'a': 1 phép so sánh
tìm 'z': 26 phép so sánh
tìm '#': 26 phép so sánh (không có, phải duyệt hết)
tổng so sánh cho cả text: 78000


Với chuỗi 26 phím, mỗi ký tự `'z'` tốn 26 phép so sánh → 3000 chữ `z` tốn 78000 phép so sánh. Với `set`, mỗi lần tra chỉ là một phép băm (trung bình `O(1)`). Không in thời gian thực vì nó phụ thuộc máy — điều cần nhớ là **độ phức tạp**.

## 6. Lời giải chính — `set` + `any`, giảng từng đoạn

**Đoạn 1 — chuẩn bị:** `broken = set(brokenLetters)`.

**Đoạn 2 — xét một từ:** `any(ch in broken for ch in word)` trả `True` nếu từ có ít nhất một ký tự hỏng. `any` tự dừng sớm như `break`.

**Đoạn 3 — đếm:** cộng 1 cho mỗi từ **không** có ký tự hỏng.

In [10]:
# Đoạn 1 + 2 trên từng từ
broken = set("lt")
for word in "leet code".split():
    has_broken = any(ch in broken for ch in word)
    print(f"{word!r}: has_broken={has_broken}")

'leet': has_broken=True
'code': has_broken=False


In [11]:
def can_type_set(text, brokenLetters):
    broken = set(brokenLetters)                   # đoạn 1
    count = 0
    for word in text.split():
        if not any(ch in broken for ch in word):  # đoạn 2
            count += 1                            # đoạn 3
    return count

print(can_type_set("hello world", "ad"))  # 1
print(can_type_set("leet code", "lt"))    # 1
print(can_type_set("leet code", "e"))     # 0
print(can_type_set("abc de", ""))         # 2 — không có phím hỏng

1
1
0
2


## 7. TRACE từng bước

Ví dụ `text = "hello world"`, `brokenLetters = "ad"` → `broken = {'a','d'}`.

| Từ | ký tự xét | `ch in broken`? | Quyết định | `count` sau |
|---|---|:---:|---|:---:|
| hello | h, e, l, l, o | đều không | gõ được | 1 |
| world | w, o, r, l | không | (tiếp tục) | 1 |
| world | d | **có** | dừng, không đếm | 1 |

Kết quả `1`. Hàm trace in đúng từng bước, và cho thấy `any` dừng ở ký tự hỏng đầu tiên.

In [12]:
def trace_can_type(text, brokenLetters):
    broken = set(brokenLetters)
    print("broken =", sorted(broken))
    count = 0
    for word in text.split():
        print(f"Từ {word!r}:")
        typed = True
        for ch in word:
            hit = ch in broken
            print(f"   ch={ch!r} in broken? {hit}")
            if hit:
                typed = False
                print("   -> gặp phím hỏng, dừng xét từ này")
                break
        if typed:
            count += 1
        print(f"   kết luận: {'gõ được' if typed else 'không gõ được'}, count = {count}")
    return count

print("Kết quả:", trace_can_type("hello world", "ad"))

broken = ['a', 'd']
Từ 'hello':
   ch='h' in broken? False
   ch='e' in broken? False
   ch='l' in broken? False
   ch='l' in broken? False
   ch='o' in broken? False
   kết luận: gõ được, count = 1
Từ 'world':
   ch='w' in broken? False
   ch='o' in broken? False
   ch='r' in broken? False
   ch='l' in broken? False
   ch='d' in broken? True
   -> gặp phím hỏng, dừng xét từ này
   kết luận: không gõ được, count = 1
Kết quả: 1


### 7.1. Trace nhánh đặc biệt: `brokenLetters` rỗng và mọi từ đều hỏng

In [13]:
print("Kết quả:", trace_can_type("hi yo", ""))
print()
print("Kết quả:", trace_can_type("leet code", "e"))

broken = []
Từ 'hi':
   ch='h' in broken? False
   ch='i' in broken? False
   kết luận: gõ được, count = 1
Từ 'yo':
   ch='y' in broken? False
   ch='o' in broken? False
   kết luận: gõ được, count = 2
Kết quả: 2

broken = ['e']
Từ 'leet':
   ch='l' in broken? False
   ch='e' in broken? True
   -> gặp phím hỏng, dừng xét từ này
   kết luận: không gõ được, count = 0
Từ 'code':
   ch='c' in broken? False
   ch='o' in broken? False
   ch='d' in broken? False
   ch='e' in broken? True
   -> gặp phím hỏng, dừng xét từ này
   kết luận: không gõ được, count = 0
Kết quả: 0


## 8. Các cách giải khác

### 8.1. Cách 3 — phép giao tập / `isdisjoint` (một dòng)

Một từ gõ được ⇔ `set(word).isdisjoint(broken)`. Kết hợp `sum` trên bool để đếm.

In [14]:
def can_type_disjoint(text, brokenLetters):
    broken = set(brokenLetters)
    return sum(set(word).isdisjoint(broken) for word in text.split())

print(can_type_disjoint("hello world", "ad"), can_type_disjoint("leet code", "lt"), can_type_disjoint("leet code", "e"))

1 1 0


### 8.2. Cách 4 — một lượt duyệt, không `split`

`split` tạo list các từ (tốn thêm `O(n)` bộ nhớ). Ta có thể duyệt thẳng từng ký tự của `text` với một cờ `word_ok`:
- gặp dấu cách → một từ vừa **kết thúc**: nếu `word_ok` thì đếm, rồi reset `word_ok = True`;
- gặp chữ hỏng → `word_ok = False`.
- Hết chuỗi → từ **cuối** chưa được chốt (không có dấu cách theo sau) → chốt thêm một lần.

Mẹo gọn: duyệt `text + " "` để từ cuối cũng có "dấu cách kết thúc" (đổi lại tạo một chuỗi mới).

In [15]:
def can_type_one_pass(text, brokenLetters):
    broken = set(brokenLetters)
    count = 0
    word_ok = True
    for ch in text:
        if ch == " ":
            count += word_ok      # chốt từ vừa xong
            word_ok = True
        elif ch in broken:
            word_ok = False
    count += word_ok              # chốt từ cuối cùng
    return count

print(can_type_one_pass("hello world", "ad"), can_type_one_pass("leet code", "lt"), can_type_one_pass("leet code", "e"))

1 1 0


### 8.3. Cách 5 — đếm từ "hỏng" rồi lấy phần bù

Số từ gõ được = tổng số từ − số từ chứa ít nhất một phím hỏng. Tổng số từ = số dấu cách + 1 (do đề đảm bảo đúng một dấu cách giữa hai từ).

In [16]:
def can_type_complement(text, brokenLetters):
    broken = set(brokenLetters)
    total = text.count(" ") + 1
    bad = sum(1 for word in text.split() if not broken.isdisjoint(word))  # isdisjoint nhận cả chuỗi
    return total - bad

print(can_type_complement("hello world", "ad"), can_type_complement("leet code", "e"))
print(set("ad").isdisjoint("hello"))   # isdisjoint chấp nhận bất kỳ iterable nào, không cần set(word)

1 0
True


## 9. Code nộp LeetCode (Python 3)

In [17]:
class Solution:
    def canBeTypedWords(self, text: str, brokenLetters: str) -> int:
        broken = set(brokenLetters)
        count = 0
        for word in text.split():
            if not any(ch in broken for ch in word):
                count += 1
        return count


sol = Solution()
assert sol.canBeTypedWords("hello world", "ad") == 1
assert sol.canBeTypedWords("leet code", "lt") == 1
assert sol.canBeTypedWords("leet code", "e") == 0
print("3 ví dụ của đề PASS")

3 ví dụ của đề PASS


Bản một dòng (cũng hợp lệ):

```python
class Solution:
    def canBeTypedWords(self, text: str, brokenLetters: str) -> int:
        broken = set(brokenLetters)
        return sum(broken.isdisjoint(word) for word in text.split())
```

## 10. Vì sao thuật toán đúng?

- `text.split()` trả về **chính xác** danh sách các từ của `text` (đề đảm bảo đúng một dấu cách giữa các từ, không dấu cách thừa), mỗi từ xuất hiện đúng một lần trong vòng lặp.
- Với mỗi từ `w`: ta gõ được `w` ⇔ gõ được **mọi** ký tự của `w` ⇔ không ký tự nào thuộc `broken`. Biểu thức `not any(ch in broken for ch in w)` đúng bằng mệnh đề đó (luật De Morgan: "không tồn tại ký tự hỏng" = "mọi ký tự đều tốt").
- **Invariant:** sau khi xử lý `k` từ đầu tiên, `count` = số từ gõ được trong `k` từ đó. Bước `k+1` chỉ cộng 1 khi từ thứ `k+1` gõ được → invariant giữ nguyên. Khi hết từ, `count` là đáp án.
- `any` dừng sớm không làm sai kết quả: một khi đã thấy một ký tự hỏng, các ký tự còn lại không thể đổi kết luận "không gõ được".

## 11. Time / Space Complexity

Gọi `n = len(text)`, `m = len(brokenLetters)` (≤ 26).

| Cách | Time | Space phụ | Ghi chú |
|---|---|---|---|
| 1. Lồng vòng, `in` trên chuỗi | `O(n·m)` | `O(n)` (list từ của `split`) | baseline dễ nghĩ |
| 2. `set` + `any` (chính) | `O(n + m)` trung bình | `O(n + m)` | rõ ràng, nên nộp |
| 3. `isdisjoint` | `O(n + m)` | `O(n + m)` (tạo `set(word)`) | ngắn |
| 4. Một lượt không `split` | `O(n + m)` | `O(m)` | tiết kiệm bộ nhớ nhất |
| 5. Phần bù | `O(n + m)` | `O(n + m)` | luyện tư duy đếm ngược |

Vì `m ≤ 26` là hằng số, có thể nói gọn mọi cách đều `O(n)`; nhưng hiểu vì sao `set` tốt hơn mới là điều mang sang bài khác.

## 12. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

In [18]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
check("đề 1", sol.canBeTypedWords("hello world", "ad"), 1)
check("đề 2", sol.canBeTypedWords("leet code", "lt"), 1)
check("đề 3", sol.canBeTypedWords("leet code", "e"), 0)
check("không phím hỏng", sol.canBeTypedWords("a bb ccc", ""), 3)
check("một từ, gõ được", sol.canBeTypedWords("x", "y"), 1)
check("một từ, hỏng", sol.canBeTypedWords("x", "x"), 0)
check("hỏng cả 26 phím", sol.canBeTypedWords("abc xyz", "abcdefghijklmnopqrstuvwxyz"), 0)
check("chữ hỏng ở cuối từ", sol.canBeTypedWords("abcd efg", "d"), 1)
check("chữ hỏng ở từ cuối", sol.canBeTypedWords("ab cd", "d"), 1)
check("từ lặp lại", sol.canBeTypedWords("go go go", "z"), 3)

PASS đề 1: 1
PASS đề 2: 1
PASS đề 3: 0
PASS không phím hỏng: 3
PASS một từ, gõ được: 1
PASS một từ, hỏng: 0
PASS hỏng cả 26 phím: 0
PASS chữ hỏng ở cuối từ: 1
PASS chữ hỏng ở từ cuối: 1
PASS từ lặp lại: 3


In [19]:
# Đối chiếu 6 cách trên dữ liệu ngẫu nhiên có seed
import random
random.seed(1935)
alphabet = "abcdefghijklmnopqrstuvwxyz"
funcs = [can_type_baseline, can_type_for_else, can_type_set, can_type_disjoint,
         can_type_one_pass, can_type_complement, Solution().canBeTypedWords]
for _ in range(500):
    words = ["".join(random.choice("abcdef") for _ in range(random.randint(1, 5)))
             for _ in range(random.randint(1, 6))]
    text = " ".join(words)
    broken = "".join(random.sample(alphabet[:8], random.randint(0, 4)))
    results = {f(text, broken) for f in funcs}
    assert len(results) == 1, (text, broken, results)
print("500 bộ dữ liệu ngẫu nhiên: mọi cách cho cùng kết quả")

500 bộ dữ liệu ngẫu nhiên: mọi cách cho cùng kết quả


## 13. Sai lầm người mới hay mắc

### Lỗi 1 — Đếm **ký tự** hỏng thay vì đếm **từ**

```python
return len(text.split()) - sum(ch in broken for ch in text)   # SAI
```
Từ `"leet"` có 2 ký tự hỏng (`l`, `t`) sẽ bị trừ 2 lần. Phải trừ theo từ.

In [20]:
def buggy_count_chars(text, brokenLetters):
    broken = set(brokenLetters)
    return len(text.split()) - sum(ch in broken for ch in text)

print("buggy:", buggy_count_chars("leet code", "lt"), "| đúng:", can_type_set("leet code", "lt"))
print("buggy:", buggy_count_chars("leet code", "e"), "| đúng:", can_type_set("leet code", "e"))

buggy: 0 | đúng: 1
buggy: -1 | đúng: 0


Bản lỗi cho `0`: từ `"leet"` có **hai** ký tự hỏng (`l`, `t`) nên bị trừ hai lần, `2 - 2 = 0`, trong khi đáp đúng là `1`. Với `"leet code"` và `"e"` (ba chữ `e`) bản lỗi còn ra số **âm** `2 - 3 = -1`. Sai lệch xuất hiện mỗi khi một từ có **nhiều hơn một** ký tự hỏng.

### Lỗi 2 — Đặt `count += 1` **bên trong** vòng ký tự

```python
for ch in word:
    if ch not in broken:
        count += 1        # SAI: đếm số ký tự tốt, không phải số từ
```

### Lỗi 3 — Quên chốt từ cuối trong cách một lượt

Nếu chỉ đếm khi gặp dấu cách, từ cuối cùng (không có dấu cách sau) bị bỏ qua.

In [21]:
def buggy_one_pass(text, brokenLetters):
    broken = set(brokenLetters)
    count, word_ok = 0, True
    for ch in text:
        if ch == " ":
            count += word_ok
            word_ok = True
        elif ch in broken:
            word_ok = False
    return count              # QUÊN: count += word_ok cho từ cuối

print("buggy:", buggy_one_pass("hello world", "a"), "| đúng:", can_type_one_pass("hello world", "a"))

buggy: 1 | đúng: 2


### Lỗi 4 — Quên reset cờ khi sang từ mới

Nếu không gán lại `word_ok = True` (hoặc `ok = True`) ở đầu mỗi từ, một từ hỏng sẽ "lây" sang mọi từ phía sau → kết quả nhỏ hơn thực tế.

### Lỗi 5 — Dùng `brokenLetters in word`

`"ad" in "hello"` kiểm tra **chuỗi con** `"ad"` liên tiếp, không phải "có chữ a hoặc chữ d". Ví dụ `"ad" in "dance"` là `False` dù `dance` chứa cả `d` lẫn `a`.

In [22]:
print("ad" in "dance")                        # False — sai ý đồ
print(any(ch in "dance" for ch in "ad"))      # True — đúng ý đồ
print("" in "hello")                           # True — chuỗi rỗng là chuỗi con của mọi chuỗi!

False
True
True


Dòng cuối là cái bẫy nữa: nếu `brokenLetters == ""` thì `"" in word` luôn `True`, khiến bản lỗi kết luận *mọi* từ đều hỏng.

## 14. Mẹo nhận dạng pattern

- "Kiểm tra phần tử có thuộc một nhóm nhỏ cố định không, lặp lại nhiều lần" → dựng `set` **một lần**, tra nhiều lần.
- "Một đối tượng hợp lệ ⇔ **mọi** thành phần hợp lệ" → `all(...)` hoặc `not any(...)`, nhớ dừng sớm.
- "Hai tập có phần tử chung không" → `isdisjoint` / phép giao `&`.
- "Xử lý theo từ/đoạn ngăn bởi ký tự phân cách" → `split`, hoặc duyệt một lượt với cờ + **chốt đoạn cuối**.
- Bài cùng họ: 1832 (Check if the Sentence Is Pangram), 771 (Jewels and Stones), 1684 (Count the Number of Consistent Strings) — bài 1684 gần như là "phần bù" của bài này.

## 15. Biến thể

### 15.1. Trả về danh sách các từ gõ được

In [23]:
def typable_words(text, brokenLetters):
    broken = set(brokenLetters)
    return [w for w in text.split() if broken.isdisjoint(w)]

print(typable_words("hello world leet code", "ad"))   # ['hello', 'leet']

['hello', 'leet']


### 15.2. LeetCode 1684 — Count the Number of Consistent Strings

Ngược lại: chỉ có các phím **trong** `allowed` là dùng được. Từ hợp lệ ⇔ mọi ký tự thuộc `allowed` ⇔ `set(word) <= allowed` (toán tử `<=` trên set nghĩa là "tập con").

In [24]:
def count_consistent(allowed, words):
    allowed = set(allowed)
    return sum(set(w) <= allowed for w in words)

print(count_consistent("ab", ["ad", "bd", "aaab", "baa", "badab"]))   # 2
print({"a"} <= {"a", "b"}, {"a", "c"} <= {"a", "b"})                   # True False

2
True False


### 15.3. Nếu được sửa đúng một phím hỏng, nên sửa phím nào?

Với mỗi phím hỏng `c`, giả sử sửa nó (bỏ khỏi `broken`) rồi đếm lại; chọn phím cho kết quả lớn nhất. Cách vét cạn này `O(m · (n + m))`, vẫn nhỏ vì `m ≤ 26`.

In [25]:
def best_key_to_fix(text, brokenLetters):
    best_key, best = None, can_type_set(text, brokenLetters)
    for c in brokenLetters:
        value = can_type_set(text, brokenLetters.replace(c, ""))
        if value > best:
            best_key, best = c, value
    return best_key, best

print(best_key_to_fix("leet code ate", "et"))   # sửa 'e' hay 't'?

('e', 1)


## 16. Bài tập (TODO — tự làm, không chạy tự động)

**Bài 1 (dự đoán output):** không chạy, cho biết `"ab cd".split(" ")`, `set("hello")` có bao nhiêu phần tử, và `all([])`.

**Bài 2 (tính tay):** `text = "the quick brown fox"`, `brokenLetters = "qx"`. Lập bảng từ / ký tự hỏng / kết luận và cho kết quả.

**Bài 3 (điền chỗ trống):** hoàn thiện baseline dùng cờ `ok`.

In [26]:
def exercise_baseline(text, brokenLetters):
    count = 0
    for word in text.split():
        # TODO 1: đặt cờ ok = True
        # TODO 2: duyệt ch trong word; nếu ch hỏng -> ok = False, break
        # TODO 3: nếu ok thì count += 1
        pass
    return count

# assert exercise_baseline("hello world", "ad") == 1
# assert exercise_baseline("leet code", "e") == 0

**Bài 4 (tự viết lại):** viết `exercise_one_pass(text, brokenLetters)` **không** dùng `split`, chỉ một vòng `for` qua từng ký tự (nhớ chốt từ cuối).

In [27]:
def exercise_one_pass(text, brokenLetters):
    # TODO
    pass

# assert exercise_one_pass("hello world", "ad") == 1
# assert exercise_one_pass("a b c", "") == 3

**Bài 5 (biến thể):** viết `exercise_count_consistent(allowed, words)` cho LeetCode 1684 (đếm từ chỉ dùng chữ trong `allowed`).

**Bài 6 (mini challenge):** viết `exercise_min_keys_to_fix(text, brokenLetters)` trả **số phím ít nhất** cần sửa để gõ được **mọi** từ. (Gợi ý: phím nào thực sự cần sửa?)

In [28]:
def exercise_count_consistent(allowed, words):
    # TODO
    pass

def exercise_min_keys_to_fix(text, brokenLetters):
    # TODO
    pass

# assert exercise_count_consistent("ab", ["ad", "bd", "aaab", "baa", "badab"]) == 2
# assert exercise_min_keys_to_fix("hello world", "adz") == 1
# assert exercise_min_keys_to_fix("abc", "xyz") == 0

## 17. Gợi ý 3 mức

| Bài | Mức 1 — hướng nghĩ | Mức 2 — cấu trúc | Mức 3 — gần code |
|---|---|---|---|
| 1 | `split(" ")` tách đúng một dấu cách | `set` bỏ trùng | `all` của dãy rỗng là "đúng một cách trống rỗng" |
| 2 | xét từng từ | từ nào chứa `q` hoặc `x`? | `quick`, `fox` hỏng |
| 3 | cờ cho từng từ | `ok = True` đặt **trong** vòng từ | `if ch in brokenLetters: ok = False; break` |
| 4 | dấu cách là "kết thúc từ" | `word_ok` + `count += word_ok` | nhớ `count += word_ok` sau vòng lặp |
| 5 | tập con | `set(w) <= allowed` | `sum(set(w) <= allowed for w in words)` |
| 6 | chỉ phím hỏng **xuất hiện** trong text mới cần sửa | giao `set(brokenLetters) & set(text)` | `len(set(brokenLetters) & set(text.replace(" ", "")))` |

## 18. Đáp án tham khảo — chỉ xem sau khi tự làm

**Bài 1:** `['ab', 'cd']`; `set("hello")` có 4 phần tử `{h, e, l, o}`; `all([])` là `True`.

**Bài 2:** `the` gõ được, `quick` hỏng (`q`), `brown` gõ được, `fox` hỏng (`x`) → kết quả `2`.

In [29]:
print("ab cd".split(" "), len(set("hello")), all([]))
print(can_type_set("the quick brown fox", "qx"))   # 2


def answer_baseline(text, brokenLetters):
    count = 0
    for word in text.split():
        ok = True
        for ch in word:
            if ch in brokenLetters:
                ok = False
                break
        if ok:
            count += 1
    return count


def answer_one_pass(text, brokenLetters):
    broken = set(brokenLetters)
    count, word_ok = 0, True
    for ch in text:
        if ch == " ":
            count += word_ok
            word_ok = True
        elif ch in broken:
            word_ok = False
    return count + word_ok


def answer_count_consistent(allowed, words):
    allowed = set(allowed)
    return sum(set(w) <= allowed for w in words)


def answer_min_keys_to_fix(text, brokenLetters):
    return len(set(brokenLetters) & set(text.replace(" ", "")))


assert answer_baseline("hello world", "ad") == 1
assert answer_baseline("leet code", "e") == 0
assert answer_one_pass("hello world", "ad") == 1
assert answer_one_pass("a b c", "") == 3
assert answer_count_consistent("ab", ["ad", "bd", "aaab", "baa", "badab"]) == 2
assert answer_min_keys_to_fix("hello world", "adz") == 1
assert answer_min_keys_to_fix("abc", "xyz") == 0
print("Đáp án tham khảo PASS toàn bộ test!")

['ab', 'cd'] 4 True
2
Đáp án tham khảo PASS toàn bộ test!


## 19. Template nhớ nhanh + checklist submit

```python
class Solution:
    def canBeTypedWords(self, text: str, brokenLetters: str) -> int:
        broken = set(brokenLetters)                       # dựng set MỘT lần
        return sum(broken.isdisjoint(w) for w in text.split())
```

**Câu thần chú:** *tách từ → từ tốt ⇔ không chung chữ nào với tập hỏng → đếm.*

**Checklist trước khi submit:**
- [ ] Đếm **từ**, không đếm ký tự?
- [ ] `brokenLetters` rỗng có trả về số từ không?
- [ ] Nếu duyệt một lượt: đã **chốt từ cuối** và **reset cờ** mỗi từ chưa?
- [ ] Không dùng `brokenLetters in word` (đó là kiểm tra chuỗi con)?
- [ ] Đã test từ có nhiều chữ hỏng (`"leet"` với `"lt"`)?